In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
columns = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
df = pd.read_csv(url, names=columns, na_values='?')

In [ ]:
print(df.head())

In [ ]:
print(df.shape)

In [ ]:
print(df.dtypes)

In [ ]:
print(df['target'].value_counts())

In [ ]:
df['target'] = (df['target']>0).astype(int)

In [ ]:
df['target'].value_counts(normalize = True)

In [ ]:
print(df.isnull().sum())

In [ ]:
#TARGET DISTRIBUTION
sns.countplot(x = 'target' , data = df)
plt.title('Heart Disease Presence (0 = No , 1 = Yes)')
plt.show()

In [ ]:
#NUMERIC FEATURE DISTRIBUTIONS
numeric_cols = ['age' , 'trestbps' , 'chol' , 'thalach' , 'oldpeak']
df[numeric_cols].hist(figsize = (14 , 6) , bins = 20)
plt.tight_layout()
plt.show()

In [ ]:
#AGE VS DISEASE PRESENCE
sns.boxplot(x = 'target' , y = 'age' , data = df)
plt.title('Age vs Disease Presence')
plt.show()

In [ ]:
#MAX HEART RATE VS DISEASE
sns.boxplot(x = 'target' , y = 'thalach' , data = df)
plt.title('Max Heart Rate vs Disease Presence')
plt.show()

In [ ]:
#CHEST PAIN TYPE VS DISEASE
pd.crosstab(df['cp'] , df['target'] , normalize = 'index').plot(kind = 'bar' , figsize = (8 , 5) , stacked = True)
plt.title('Chest Pain Type vs Disease Presence')
plt.xlabel('Chest Pain Type')
plt.ylabel('Presence Probability')
plt.show()

In [ ]:
#CORRELATION HEATMAP
plt.figure(figsize=(8 , 6))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
#FILL MISSING VALUES WITH MEDIAN
df['ca'] = df['ca'].fillna(df['ca'].median())
df['thal'] = df['thal'].fillna(df['thal'].median())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

x = df.drop('target' , axis = 1)
y = df['target']

x_train , x_test , y_train , y_test = train_test_split(x , y , test_size = 0.2 , random_state = 42 , stratify = y)

In [ ]:
print(x_train.shape, x_test.shape)

In [ ]:
print(y_train.value_counts(normalize = True))

In [ ]:
print(y_test.value_counts(normalize = True))

In [ ]:
categorical_cols = ['cp', 'restecg', 'slope', 'thal']
x = pd.get_dummies(x, columns=categorical_cols, drop_first=True)

In [ ]:
scaler = StandardScaler()

x_train_scaled = x_train.copy()
x_test_scaled = x_test.copy()

numeric_cols_to_scale = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
x_train_scaled[numeric_cols_to_scale] = scaler.fit_transform(x_train[numeric_cols_to_scale])
x_test_scaled[numeric_cols_to_scale] = scaler.transform(x_test[numeric_cols_to_scale])

In [ ]:
print(x_train_scaled.shape)

In [ ]:
#MODELS IMPORT
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
)

In [ ]:
#LOGISTIC REGRESSION
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(x_train_scaled, y_train)
y_pred_lr = log_reg.predict(x_test_scaled)
y_proba_lr = log_reg.predict_proba(x_test_scaled)[:, 1]

In [ ]:
#SVM
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(x_train_scaled, y_train)
y_pred_svm = svm.predict(x_test_scaled)
y_proba_svm = svm.predict_proba(x_test_scaled)[:, 1]


In [ ]:
# 3. Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(x_train_scaled, y_train)
y_pred_rf = rf.predict(x_test_scaled)
y_proba_rf = rf.predict_proba(x_test_scaled)[:, 1]

In [ ]:
#XG BOOST
from xgboost import XGBClassifier
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(x_train_scaled, y_train)
y_pred_xgb = xgb.predict(x_test_scaled)
y_proba_xgb = xgb.predict_proba(x_test_scaled)[:, 1]

In [ ]:
results = {
    'Logistic Regression': (y_pred_lr, y_proba_lr),
    'SVM': (y_pred_svm, y_proba_svm),
    'Random Forest': (y_pred_rf, y_proba_rf),
    'XGBoost': (y_pred_xgb, y_proba_xgb),
}

print(f"{'Model':<22}{'Accuracy':<10}{'Precision':<11}{'Recall':<9}{'F1':<8}{'ROC-AUC':<8}")
for name, (pred, proba) in results.items():
    print(f"{name:<22}{accuracy_score(y_test, pred):<10.3f}"
          f"{precision_score(y_test, pred):<11.3f}"
          f"{recall_score(y_test, pred):<9.3f}"
          f"{f1_score(y_test, pred):<8.3f}"
          f"{roc_auc_score(y_test, proba):<8.3f}")

# ROC curves
plt.figure(figsize=(7, 6))
for name, (pred, proba) in results.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()

In [ ]:
#TUNING RANDOM FOREST
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(x_train_scaled, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)

In [ ]:
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(x_test_scaled)
y_proba_tuned = best_rf.predict_proba(x_test_scaled)[:, 1]

print("Tuned Random Forest — Test Set Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("Precision:", precision_score(y_test, y_pred_tuned))
print("Recall:", recall_score(y_test, y_pred_tuned))
print("F1 Score:", f1_score(y_test, y_pred_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_tuned))

In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=x_train_scaled.columns)

plt.figure(figsize=(8, 6))
importances.sort_values(ascending=False).head(10).plot(kind='barh')
plt.title('Top 10 Feature Importances (Tuned Random Forest)')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
import joblib
joblib.dump()